# BERT LLM Hallucination Detector

### Imports

In [1]:
import os
import json
import time
import ast
import unicodedata
from datetime import datetime
import random
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from dotenv import load_dotenv
from tavily import TavilyClient
from langdetect import detect, DetectorFactory
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

### Step 1: Initialization and Setup

In [2]:
load_dotenv()
API_Tavily = os.getenv("API_Tavily")
api_keys = [
        os.getenv("GROQ_API_KEY1"),
        os.getenv("GROQ_API_KEY2"),
        os.getenv("GROQ_API_KEY3"),
        os.getenv("GROQ_API_KEY4"),
        os.getenv("GROQ_API_KEY5"),
        os.getenv("GROQ_API_KEY6"),
        os.getenv("GROQ_API_KEY7"),
        os.getenv("GROQ_API_KEY8"),
        os.getenv("GROQ_API_KEY9"),
        os.getenv("GROQ_API_KEY10"),
        os.getenv("GROQ_API_KEY11"),
        os.getenv("GROQ_API_KEY12"),
        os.getenv("GROQ_API_KEY13"),
        os.getenv("GROQ_API_KEY14"),
        os.getenv("GROQ_API_KEY15"),
        os.getenv("GROQ_API_KEY16"),
        os.getenv("GROQ_API_KEY17")
    ]
api_keys = [k for k in api_keys if k]
DetectorFactory.seed = 0

# Initialize clients and tokenizer
client = TavilyClient(API_Tavily)
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-uncased")

# Initialize Translation/Groq LLM
model = ChatGroq(
    api_key=api_keys[0],
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_retries=2,
)

### Step 2: Data Loading

In [3]:
dataset = load_dataset("Helsinki-NLP/mu-shroom", "all")

# Access splits
train = dataset["train_unlabeled"]
val = dataset["validation"]
test = dataset["test"]

# add indices to the dataset
def add_id(example, idx):
    example["id"] = idx
    return example

# normalize the characters in the dataset to avoid issues with special characters
def normalize_text(text):
    if text is None:
        return ""

    # Normalize Unicode representation
    text = unicodedata.normalize("NFC", text)

    # Normalize quotation marks/apostrophes
    replacements = {
        "’": "'",
        "‘": "'",
        "“": '"',
        "”": '"',
        "„": '"',
        "‹": "'",
        "›": "'",
        "\u00A0": " ",  # non-breaking space
        "\u200B": "",   # zero-width space
        "\u200D": "",   # zero-width joiner
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    return text.strip()


train = train.map(
    add_id,
    with_indices=True
)

### Step 3: Translation and Normalization Logic

In [4]:
def detect_language(text):
    """
    Detect language using ISO-639-1 codes:
    en, de, fr, hi, etc.
    """
    try:
        return detect(text)
    except Exception:
        return "unknown"

def translate_text(text, target_language):
    messages = [
    ("system", f"You are a helpful translator. Translate the user sentence to {target_language}."),
    ("human", text),
    ]
    response = model.invoke(messages)
    return response.content.strip()

def normalize_answer_language(question, answer):
    """
    Check if question and answer languages match.
    Translate answer if needed.
    """
    question_lang = detect_language(question)
    answer_lang = detect_language(answer)

    if question_lang == answer_lang:
        return answer

    return translate_text(
        answer,
        question_lang
    )

def normalize_special_characters(example):

    example["model_input"] = normalize_text(
        example["model_input"]
    )

    example["model_output_text"] = normalize_text(
        example["model_output_text"]
    )

    example["reference_answer"] = normalize_text(
        example["reference_answer"]
    )

    return example

### Step 4: Ground Truth Retrieval (Tavily)

In [5]:
unique_questions = list(set(train["model_input"]))

# Get reference first old else new
if os.path.exists("reference_map.json"):
    with open("reference_map.json", "r", encoding="utf-8") as f:
        reference_map = json.load(f)
else:
    reference_map = {}

# Search questions
for question in unique_questions:
    # Skip already searched questions
    if question in reference_map:
        continue

    # Get answers
    try:
        response = client.search(
            query=question,
            include_answer="basic",
            search_depth="basic"
        )
        answer = response.get("answer") or "No answer returned"
        answer = normalize_answer_language(question, answer)
        reference_map[question] = answer
        
        # Save immediately after each successful search
        with open("reference_map.json", "w", encoding="utf-8") as f:
            json.dump(reference_map, f, indent=4, ensure_ascii=False)
            
    except Exception as e:
        continue


### Step 5: Map References to Dataset

In [6]:
def add_reference(example):
    example["reference_answer"] = reference_map[example["model_input"]]
    return example

train = train.map(add_reference)
train = train.map(normalize_special_characters)

In [7]:
train[1000]

{'id': 1000,
 'lang': 'ES',
 'model_input': '¿Qué tipo de bebida es el Capitan Morgan?',
 'model_output_text': 'Captain Morgan es una marca de ron, que es un tipo de bebida alcohólica destilada hecha a partir de la caña de azúcar o la remolacha azucarada. Se originó en Gales y lleva el nombre del pirata del siglo XVII, Sir Henry Morgan. La marca es conocida por sus sabores distintivos y su fuerte contenido de alcohol.',
 'model_id': 'Iker/Llama-3-Instruct-Neurona-8b-v2',
 'wikipedia_url': None,
 'soft_labels': None,
 'hard_labels': None,
 'model_output_logits': [24.1562767029,
  29.8636245728,
  25.7136325836,
  26.4448757172,
  22.5440921783,
  24.1393051147,
  19.9062595367,
  32.0728797913,
  18.5324344635,
  20.3986377716,
  25.8859920502,
  26.9128742218,
  22.8077888489,
  26.4061126709,
  23.6107997894,
  36.1078109741,
  23.5410652161,
  27.2813053131,
  30.6391525269,
  30.9397315979,
  19.7266635895,
  28.3470802307,
  31.5850448608,
  20.1085243225,
  35.6496505737,
  22.993

### Step 6: LLM Hallucination Detection Prompting

#### 6.1 Prompt Templates

In [8]:
def prompt_template(model_input,model_output_text,reference_answer):
    """
    Generate 5 prompts for hallucination detection.
    params: model_input, model_output_text, reference_answer
    return: list of 5 prompts
    """
    # default
    message1 = [
    (
    "system",
    "You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
    ),
    (
    "user",
    f"""
    Compare the LLM answer with the reference answer.

    Return ONLY a Python list of strings:
    ["hallucinated_span_1", "hallucinated_span_2"]

    Rules:
    - hallucinated_span must be copied EXACTLY from the LLM answer.
    - Do not paraphrase or correct the text.
    - Include only false or unsupported factual claims.
    - Include complete factual claims that are false or unsupported.
    - The spans must be as short as possible, but include the complete factual claim that is false.
    - If no hallucination exists, return [].
    - Do not provide explanations.

    Question:
    {model_input}

    LLM Answer:
    {model_output_text}

    Reference Answer:
    {reference_answer}
    """
    )
    ]
    # Atomic claim decomposition
    message2 = [
    (
    "system",
    "You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
    ),
    (
    "user",
    f"""
    Compare the LLM answer with the reference answer.

    Return ONLY a Python list of strings:
    ["hallucinated_span_1", "hallucinated_span_2"]

    Rules:
    - Copy spans exactly from the LLM answer.
    - Identify each individual false factual claim separately.
    - If a sentence contains multiple hallucinated facts, return separate spans.
    - Do not include correct parts of the sentence.
    - Do not mark equivalent statements as hallucinations.
    - If no hallucination exists, return [].
    - Do not provide explanations.

    Question:
    {model_input}

    LLM Answer:
    {model_output_text}

    Reference Answer:
    {reference_answer}
    """
    )
    ]
    # Strict contradiction detector
    message3 = [
    (
    "system",
    "You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
    ),
    (
    "user",
    f"""
    Compare the LLM answer with the reference answer.

    Return ONLY a Python list of strings:
    ["hallucinated_span_1", "hallucinated_span_2"]

    Rules:
    - Copy spans exactly from the LLM answer.
    - Only return claims that directly contradict the reference answer.
    - Do not mark missing information as hallucination.
    - Do not mark different wording with the same meaning as hallucination.
    - Avoid guessing.
    - If no direct contradiction exists, return [].
    - Do not provide explanations.

    Question:
    {model_input}

    LLM Answer:
    {model_output_text}

    Reference Answer:
    {reference_answer}
    """
    )
    ]
    # Complete factual claim
    message4 = [
    (
    "system",
    "You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
    ),
    (
    "user",
    f"""
    Compare the LLM answer with the reference answer.

    Return ONLY a Python list of strings:
    ["hallucinated_span_1", "hallucinated_span_2"]

    Rules:
    - Copy spans exactly from the LLM answer.
    - Return complete factual claims that are false or unsupported.
    - Include enough context to understand why the claim is wrong.
    - Do not return isolated words or numbers.
    - Do not mark correct paraphrases as hallucinations.
    - If no hallucination exists, return [].
    - Do not provide explanations.

    Question:
    {model_input}

    LLM Answer:
    {model_output_text}

    Reference Answer:
    {reference_answer}
    """
    )
    ]
    # smallest hall
    message5 = [
    (
    "system",
    "You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
    ),
    (
    "user",
    f"""
    Compare the LLM answer with the reference answer.

    Return ONLY a Python list of strings:
    ["hallucinated_span_1", "hallucinated_span_2"]

    Rules:
    - Copy spans exactly from the LLM answer.
    - Return only statements that are factually incorrect or unsupported.
    - Select the smallest span that contains the hallucinated information.
    - Do not include surrounding correct information.
    - Do not mark paraphrases or equivalent meanings as hallucinations.
    - If no hallucination exists, return [].
    - Do not provide explanations.

    Question:
    {model_input}

    LLM Answer:
    {model_output_text}

    Reference Answer:
    {reference_answer}
    """
    )
    ]
    # Context-aware balanced annotation
    message6 = [(
    "system",
    "You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
    ),
    (
    "user",
    f"""
    Compare the LLM answer with the reference answer.

    Return ONLY a Python list of strings:
    ["hallucinated_span_1", "hallucinated_span_2"]

    Rules:
    - Copy spans exactly from the LLM answer.
    - Return the shortest span that still represents a complete false factual claim.
    - Consider the meaning of the statement, not only exact wording.
    - Do not mark correct explanations, paraphrases, or logical consequences of the reference answer.
    - Do not include unsupported speculation unless it conflicts with the reference answer.
    - If no hallucination exists, return [].
    - Do not provide explanations.

    Question:
    {model_input}

    LLM Answer:
    {model_output_text}

    Reference Answer:
    {reference_answer}
    """
    )
    ]

    # get 2 random messages from the 6 messages
    messages = random.sample([message1,message2,message3,message4,message5,message6], 2)
    # messages = [message1,message2,message3,message4,message5,message6]

    return messages

#### 6.2 Model setup

In [9]:
import itertools

def create_llm_models():
    """
    Build one ChatGroq client per (model, api_key) pair and return a
    round-robin cycler for each model.

    Returns: {model_name: itertools.cycle([(key_id, client), ...])}
    """
    if not api_keys:
        raise RuntimeError("No GROQ_API_KEY* values found in .env")

    model_configs = {
        "qwen3.6-27b": dict(
            model="qwen/qwen3.6-27b",
            temperature=0,
            max_tokens=None,
            reasoning_effort="none",
            reasoning_format="hidden",
            timeout=None,
            max_retries=2,
        ),
        "llama-3.3-70b": dict(
            model="llama-3.3-70b-versatile",
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2,
        ),
        "gpt-oss-120b": dict(
            model="openai/gpt-oss-120b",
            temperature=0,
            max_tokens=None,
            reasoning_effort="low",
            reasoning_format="hidden",
            timeout=None,
            max_retries=2,
        ),
    }

    models = {}

    for name, config in model_configs.items():

        clients = [
            (i + 1, ChatGroq(api_key=key, **config))
            for i, key in enumerate(api_keys)
        ]

        # stagger each model's starting key so the three models are
        # never hitting the same key at the same moment
        offset = len(models) % len(api_keys)
        clients = clients[offset:] + clients[:offset]

        models[name] = itertools.cycle(clients)

    return models


Helper functions for LLM hallucination detection

In [10]:
def parse_spans(output):
    """
    Convert LLM response into list of hallucinated spans.
    example: string from llm('["span1", "span2"]') -> ["span1", "span2"]
    """

    try:
        spans = ast.literal_eval(output.strip())
    except Exception:
        return []

    cleaned = []

    for span in spans:

        # if model returns tuple accidentally
        if isinstance(span, (tuple, list)):
            span = span[0]

        if isinstance(span, str):
            cleaned.append(span)

    return cleaned

def spans_to_character_labels(text, spans):
    """
    Convert list of hallucinated spans into character-level labels.
    ex: ["span1", "span2"] -> [0, 0, 1, 1, 0, 0]
    """
    labels = np.zeros(len(text), dtype=np.int8)

    for span in spans:

        start = text.find(span)

        if start == -1:
            continue

        end = start + len(span)

        labels[start:end] = 1

    return labels



OUTPUT_FILE = "annotations.jsonl"
LOG_FILE = "annotation.log"


def log(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {message}\n")

    print(message)


def load_completed_samples():

    completed = set()

    if os.path.exists(OUTPUT_FILE):

        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:

            for line in f:

                try:
                    completed.add(json.loads(line)["sample_id"])
                except Exception:
                    pass

    return completed


def annotate_sample(sample_id, sample, llm_models):

    prompts = prompt_template(
        sample["model_input"],
        sample["model_output_text"],
        sample["reference_answer"]
    )

    metadata = []

    # at least one full pass through every key before giving up
    max_retry = max(5, len(api_keys))

    for model_name, key_cycle in llm_models.items():

        for prompt_id, prompt in enumerate(prompts):

            for attempt in range(max_retry):

                # new key on every attempt, including the first
                key_id, llm = next(key_cycle)

                try:
                    response = llm.invoke(prompt)
                    spans = parse_spans(response.content)

                    metadata.append({
                        "model": model_name,
                        "prompt": prompt_id,
                        "spans": spans
                    })

                    log(
                        f"Sample {sample_id} | {model_name} | key {key_id} | "
                        f"Prompt {prompt_id} | SUCCESS | {len(spans)} spans"
                    )
                    break

                except Exception as e:
                    msg = str(e)

                    log(
                        f"Sample {sample_id} | {model_name} | key {key_id} | "
                        f"Prompt {prompt_id} | FAILED | attempt {attempt + 1} | {msg}"
                    )

                    if attempt == max_retry - 1:
                        log(
                            f"Sample {sample_id} | {model_name} | key {key_id} | "
                            f"Prompt {prompt_id} | FAILED | giving up after {max_retry} attempts"
                        )

                    elif (attempt + 1) % len(api_keys) == 0:
                        # every key failed in a row -> genuinely rate limited
                        wait_time = 60
                        log(f"All {len(api_keys)} keys failed; sleeping {wait_time}s...")
                        time.sleep(wait_time)

                    else:
                        # just move on to the next key
                        time.sleep(1)

            # stay safely below Groq RPM limits
            time.sleep(0.5)

    return {
        "sample_id": sample_id,
        "model_input": sample["model_input"],
        "model_output_text": sample["model_output_text"],
        "reference_answer": sample["reference_answer"],
        "metadata": metadata
    }


In [11]:
cl =parse_spans(
    '["The capital of France is Berlin.", " The Eiffel Tower is located in New York City."]'
)
span_labels = spans_to_character_labels(
    "I like The capital of France is Berlin. The Eiffel Tower is located in New York City.", cl)
print(span_labels)
## TODO: Understand this

[0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1]


#### 6.3 Annotation of soft and hard labels for llm output

In [12]:
# def annotate_sample(sample, llm_models):
#     """
#     Annotate a sample with hallucination detection labels using multiple LLM models and prompts.
#     input: sample (dict), llm_models (dict)
#     output: dict with text, soft_labels, hard_labels, and metadata
#     """

#     text = sample["model_output_text"]

#     prompts = prompt_template(
#         sample["model_input"],
#         sample["model_output_text"],
#         sample["reference_answer"]
#     )

#     all_character_labels = []

#     metadata = []


#     for model_name, llm in llm_models.items():

#         for prompt_id, prompt in enumerate(prompts):

#             response = llm.invoke(prompt)

#             spans = parse_spans(response.content)

#             labels = spans_to_character_labels(
#                 text,
#                 spans
#             )

#             all_character_labels.append(labels)


#             metadata.append({
#                 "model": model_name,
#                 "prompt": prompt_id,
#                 "spans": spans
#             })


#     # shape:
#     # (18, number_of_characters)
#     label_matrix = np.stack(all_character_labels)


#     soft_labels = label_matrix.mean(axis=0)

#     hard_labels = (
#         soft_labels >= 0.5
#     ).astype(np.int8)


#     return {
#         "text": text,
#         "soft_labels": soft_labels.tolist(),
#         "hard_labels": hard_labels.tolist(),
#         "metadata": metadata
#     }

### 7. Finally find all annotations for the entire Dataset

In [13]:
llm_models = create_llm_models()

completed = load_completed_samples()

for sample in train:
    sample_id = sample["id"]
    if sample_id in completed:
        print(f"Skipping sample {sample_id}")
        continue

    print(f"Processing sample {sample_id}")

    result = annotate_sample(
        sample_id,
        sample,
        llm_models
    )

    with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
        f.write(
            json.dumps(
                result,
                ensure_ascii=False
            ) + "\n"
        )

    log(f"Saved sample {sample_id}")

Skipping sample 0
Skipping sample 1
Skipping sample 2
Skipping sample 3
Skipping sample 4
Skipping sample 5
Skipping sample 6
Skipping sample 7
Skipping sample 8
Skipping sample 9
Skipping sample 10
Skipping sample 11
Skipping sample 12
Skipping sample 13
Skipping sample 14
Skipping sample 15
Skipping sample 16
Skipping sample 17
Skipping sample 18
Skipping sample 19
Skipping sample 20
Skipping sample 21
Skipping sample 22
Skipping sample 23
Skipping sample 24
Skipping sample 25
Skipping sample 26
Skipping sample 27
Skipping sample 28
Skipping sample 29
Skipping sample 30
Skipping sample 31
Skipping sample 32
Skipping sample 33
Skipping sample 34
Skipping sample 35
Skipping sample 36
Skipping sample 37
Skipping sample 38
Skipping sample 39
Skipping sample 40
Skipping sample 41
Skipping sample 42
Skipping sample 43
Skipping sample 44
Skipping sample 45
Skipping sample 46
Skipping sample 47
Skipping sample 48
Skipping sample 49
Skipping sample 50
Skipping sample 51
Skipping sample 52
Ski

<unknown>:1: SyntaxWarning: invalid escape sequence '\*'


Sample 3281 | llama-3.3-70b | key 7 | Prompt 0 | SUCCESS | 5 spans
Sample 3281 | llama-3.3-70b | key 8 | Prompt 1 | FAILED | attempt 1 | Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kykdj9qwe089rm7h41weqbmq` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99549, Requested 849. Please try again in 5m43.872s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Sample 3281 | llama-3.3-70b | key 9 | Prompt 1 | FAILED | attempt 2 | Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kykdmhgzeh5rsh9pk0bjzdc7` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99576, Requested 849. Please try again in 6m7.2s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rat

<unknown>:1: SyntaxWarning: invalid escape sequence '\*'


Sample 3281 | llama-3.3-70b | key 10 | Prompt 1 | SUCCESS | 4 spans
Sample 3281 | gpt-oss-120b | key 17 | Prompt 0 | SUCCESS | 5 spans
Sample 3281 | gpt-oss-120b | key 1 | Prompt 1 | SUCCESS | 0 spans
Saved sample 3281
Processing sample 3282
Sample 3282 | qwen3.6-27b | key 15 | Prompt 0 | SUCCESS | 15 spans
Sample 3282 | qwen3.6-27b | key 16 | Prompt 1 | SUCCESS | 6 spans
Sample 3282 | llama-3.3-70b | key 11 | Prompt 0 | SUCCESS | 6 spans
Sample 3282 | llama-3.3-70b | key 12 | Prompt 1 | FAILED | attempt 1 | Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kykjfwjnf78aacv76qs1ey19` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99719, Requested 1047. Please try again in 11m1.824s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Sample 3282 | llama-3.3-70b | key 13 | Prompt 1 | FAILED | attempt 2 | Error 